# Пілот прогнозування наслідків корекції
8 кадрів × 4 рівні prior × 3 умови × 3 повтори = **288 спроб**.
Відкрийте в Colab із CUDA GPU. Потрібні каталоги `scripts/correction_prediction`, `scripts/visibility_localization` та `scripts/__init__.py` з цього репозиторію. Дані AnyVisLoc уже мають бути розпаковані.
Використовується CSV вибірки попереднього quick visibility експерименту; шляхи в ньому мають існувати в поточному Drive. Відсутні кадри не замінюються.
Нові зіставлення вимкнені до встановлення RUN_LOCALIZATION=True. Виклики Jev за замовчуванням лише з кешу. Вісім кадрів не є достатнім основним навчальним набором.

In [1]:
import os, sys
if os.environ.get('CUBLAS_WORKSPACE_CONFIG') not in (':4096:8', ':16:8') and 'torch' in sys.modules and sys.modules['torch'].cuda.is_initialized():
    raise RuntimeError('Виконайте Runtime → Restart session: CUDA вже ініціалізовано без налаштування детермінованого cuBLAS.')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, os
REPO_ROOT = Path('/content/drive/MyDrive/PhD/proj')
EXPERIMENT = REPO_ROOT/'scripts/correction_prediction'
# Укажіть реальний CSV попереднього quick visibility запуску:
SELECTION_CSV = Path(
    '/content/drive/MyDrive/PhD/visibility_results/'
    'author-visibility-quick-v1/selection/selected_samples.csv'
)
CAMP_CHECKPOINT = None  # Автоматично: кеш або завантаження офіційних ваг CAMP; можна вказати власний шлях
OUTPUT = EXPERIMENT/'results/pilot-v2-cublas'
RUNTIME_ROOT = EXPERIMENT/'runtime'
INSTALL_DEPENDENCIES = False
RUN_LOCALIZATION = True
JEV_MAX_REQUESTS = 0
sys.path.insert(0, str(REPO_ROOT))
assert (EXPERIMENT/'campaign.py').is_file(), 'Спочатку скопіюйте пакет у Drive'
os.environ['VISIBILITY_MODEL_CACHE'] = str(EXPERIMENT/'model_cache')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Середовище
У новому runtime встановіть INSTALL_DEPENDENCIES=True та виконайте наступну комірку. Вона зупинить запуск, якщо pip змінив уже імпортовані бібліотеки: виконайте Runtime → Restart session, встановіть INSTALL_DEPENDENCIES=False і запустіть notebook зверху. Перевірка SciPy виконується саме в ядрі notebook, а не лише в окремому процесі тестів. CUDA/PyTorch використовують ті самі версії, що й наявний visibility експеримент.

In [2]:
from pathlib import Path
from scripts.visibility_localization.colab import stage_archive

ARCHIVES_ROOT = Path('/content/drive/MyDrive/PhD/proj/AnyVisLoc')
DATASET_ROOT = Path('/content/visibility_localization/data')

for scene in ['Scene_05', 'Scene_08', 'Scene_11', 'Scene_21']:
    archive = ARCHIVES_ROOT / f'{scene}.zip'
    assert archive.is_file(), f'Не знайдено архів: {archive}'

    print('Розпаковуємо:', scene, flush=True)
    stage_archive(archive, DATASET_ROOT)

print('Дані розпаковано')

Розпаковуємо: Scene_05
Розпаковуємо: Scene_08
Розпаковуємо: Scene_11
Розпаковуємо: Scene_21
Дані розпаковано


In [3]:
import subprocess
from scripts.correction_prediction.environment import loaded_snapshot, require_clean_kernel, verify_scipy_imports
_CORRECTION_ENVIRONMENT_READY = False
loaded_before_install = loaded_snapshot()
if INSTALL_DEPENDENCIES:
    if not (3, 10) <= sys.version_info[:2] <= (3, 13):
        raise RuntimeError('Підтримано Python 3.10–3.13')
    torch_v, vision_v, cuda = ('2.6.0', '0.21.0', 'cu126') if sys.version_info[:2] == (3, 13) else ('2.5.1', '0.20.1', 'cu121')
    subprocess.run([sys.executable, '-m', 'pip', 'install', f'torch=={torch_v}', f'torchvision=={vision_v}', '--index-url', f'https://download.pytorch.org/whl/{cuda}'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(EXPERIMENT/'requirements-colab.txt')], check=True)
    require_clean_kernel(loaded_before_install)
    print('Залежності встановлено.')
else:
    print('Встановлення пропущено.')
require_clean_kernel()
verify_scipy_imports()
_CORRECTION_ENVIRONMENT_READY = True
print('SciPy успішно імпортується в поточному ядрі notebook.')

Встановлення пропущено.
SciPy успішно імпортується в поточному ядрі notebook.


In [4]:
import json
from scripts.correction_prediction.campaign import pilot_config, fixed_pilot_selection, run_pilot
CONFIG = pilot_config()
print(json.dumps(CONFIG, indent=2))
if SELECTION_CSV is not None:
    selection = fixed_pilot_selection(SELECTION_CSV)
    for row in selection:
        assert Path(row['path']).is_file() and Path(row['reference_json']).is_file()
        print(row['scene_id'], row['sample_id'])
test = subprocess.run([sys.executable, '-m', 'pytest', str(EXPERIMENT/'tests'), '-q', '-p', 'no:cacheprovider', '--basetemp', str(EXPERIMENT/'.colab-test-tmp')], cwd=REPO_ROOT, capture_output=True, text=True)
print(test.stdout, test.stderr)
assert test.returncode == 0, 'Спочатку виправте середовище/код; локалізацію не запускати'

{
  "seed": 42,
  "reference_mode": "aerial",
  "matching_method": "Roma",
  "map_alignment": "prior_yaw",
  "scenes": [
    "Scene_05",
    "Scene_08",
    "Scene_11",
    "Scene_21"
  ],
  "nadir_max_deg": 15.0,
  "preferred_nadir_deg": 10.0,
  "sample_count": 8,
  "pose_convention": "ned_attitude",
  "sigma_xy_m": [
    5.0,
    20.0,
    50.0,
    100.0
  ],
  "trials": 3,
  "profiles": {
    "small": {
      "yaw_deg": 2.0,
      "tilt_deg": 1.0,
      "z_m": 2.0
    },
    "main": {
      "yaw_deg": 5.0,
      "tilt_deg": 2.0,
      "z_m": 5.0
    },
    "large": {
      "yaw_deg": 15.0,
      "tilt_deg": 5.0,
      "z_m": 10.0
    }
  },
  "active_profiles": [
    "main"
  ],
  "retrieval_modes": [
    "prior"
  ],
  "top_k": 5,
  "search_sigma": 3.0,
  "gallery_overlap": 0.5,
  "patch_scale": 1.5,
  "retrieval_size": 384,
  "retrieval_batch": 8,
  "max_gallery_tiles": 12000,
  "max_patch_pixels": 4000000,
  "max_image_side": 1600,
  "match_keypoints": 3000,
  "fallback_patch_si

## Виконання пілота
Перевірте вибірку й шляхи, встановіть RUN_LOCALIZATION=True. Виберіть CUDA GPU у меню Runtime → Change runtime type → GPU; CPU-сеанс не підходить. CAMP_CHECKPOINT=None автоматично використовує кеш або завантажує офіційні ваги CAMP у EXPERIMENT/model_cache. Явний шлях потрібен лише для власної копії ваг. Ініціалізація також отримує зафіксований upstream, RoMa та DINOv2; старий notebook не потрібний. Результати зберігаються після кожної спроби; повторний запуск перевіряє ідентичність коду, даних та ваг і продовжує лише незавершені спроби. Технічні помилки зберігаються; їх виправлення потребує нового запуску з новим каталогом.

In [5]:
if RUN_LOCALIZATION:
    assert globals().get('_CORRECTION_ENVIRONMENT_READY', False), 'Спочатку виконайте комірку середовища; після зміни бібліотек потрібен Restart session.'
    import torch
    assert torch.cuda.is_available(), 'Виберіть Runtime → Change runtime type → GPU та підключіть новий сеанс; CPU не підтримується.'
    assert SELECTION_CSV is not None, 'Укажіть CSV попередньої вибірки.'
    # CAMP_CHECKPOINT=None: initialize_backend сам отримує ваги CAMP із кешу або офіційного джерела.
    from scripts.correction_prediction.colab import initialize_backend
    backend = initialize_backend(RUNTIME_ROOT, CAMP_CHECKPOINT)
    summary = run_pilot(SELECTION_CSV, backend, OUTPUT)
    print(json.dumps(summary, indent=2))
else:
    print('Зіставлення не виконувалося. RUN_LOCALIZATION=False.')

CAMP: reusing cached checkpoint /content/drive/MyDrive/PhD/proj/scripts/correction_prediction/model_cache/weights_0.9446_for_U1652.pth


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/drive/MyDrive/PhD/proj/scripts/correction_prediction/runtime/Benchmark/Retrieval_Models/CAMP/hand_convnext/ConvNext/backbones/model_convnext.py:166: UserWarning: Overwriting convnext_tiny in registry with Retrieval_Models.CAMP.hand_convnext.ConvNext.backbones.model_convnext.convnext_tiny. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/drive/MyDrive/PhD

===========building convnext===========
using model_type: convnext_base as a backbone


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Using coarse resolution (560, 560), and upsample res (864, 864)


Image Retrieval:   0%|          | 0/1 [00:00<?, ?it/s]/content/drive/MyDrive/PhD/proj/scripts/correction_prediction/runtime/Benchmark/Retrieval_Models/feature_extract.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Image Retrieval: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]
/content/drive/MyDrive/PhD/proj/scripts/correction_prediction/runtime/Benchmark/Retrieval_Models/feature_extract.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.13/dist-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Image Retrieval:   0%|    

{
  "expected_attempts": 288,
  "completed_attempts": 288,
  "technical_errors": 0,
  "unsupported_queries": 0,
  "outcomes": {
    "gain": 47,
    "gain_small": 36,
    "neutral": 3,
    "gain_large": 171,
    "reject": 27,
    "harm_small": 3,
    "harm": 1
  },
  "outcomes_exclude_final_test": true,
  "measured_localization_s": 3899.2233531700094,
  "main_campaign_approved": false,
  "validation_passed": false
}


In [6]:
DATASET = OUTPUT/'dataset.json'
if DATASET.is_file():
    from scripts.correction_prediction.dataset import load_dataset
    from collections import Counter
    data = load_dataset(DATASET)
    print('Унікальні кадри:', len({(r['scene_id'], r['sample_id']) for r in data['rows']}))
    print('Спроби:', len(data['rows']))
    print(Counter(r['label'].get('class_name', 'technical_error') for r in data['rows']))
    print('Непідтримувані запити:', sum(r['features'] is None for r in data['rows']))
    print('Технічні помилки:', sum(r['label']['technical_error'] for r in data['rows']))
else:
    print('Повного dataset.json ще немає; перевіряємо збережені окремі спроби.')
if (OUTPUT/'records').is_dir():
    from scripts.correction_prediction.diagnostics import diagnose_campaign
    diagnostic = diagnose_campaign(OUTPUT)
    print(json.dumps(diagnostic, ensure_ascii=False, indent=2))

Унікальні кадри: 8
Спроби: 288
Counter({'gain_large': 171, 'gain': 47, 'gain_small': 36, 'reject': 27, 'neutral': 3, 'harm_small': 3, 'harm': 1})
Непідтримувані запити: 0
Технічні помилки: 0


ModuleNotFoundError: No module named 'scripts.correction_prediction.diagnostics'

## Jev: окрема перевірка API та кешування
Це готова модель; ваги Jev не навчаються. Для мережевого запуску погодьте бюджет, задайте JEV_MAX_REQUESTS, а ключ внесіть у Colab Secrets під назвою TYPESAFE_API_KEY. Ліміт рахує також попередні зарезервовані/невдалі запити в цьому кеші. Один запуск варіанта може потребувати до 288 запитів; точні однакові запити використовують кеш. Не вводьте ключ у текст notebook. API-помилка не перетворюється на відмову локалізатора. Зображення — фрагмент карти, не кадр камери.

In [ ]:
RUN_JEV = False
WITH_MAP_IMAGE = False
if RUN_JEV:
    if JEV_MAX_REQUESTS > 0:
        from google.colab import userdata
        os.environ['TYPESAFE_API_KEY'] = userdata.get('TYPESAFE_API_KEY')
    from scripts.correction_prediction.jev_campaign import run_jev_predictions
    variant = 'map' if WITH_MAP_IMAGE else 'numeric'
    print(run_jev_predictions(DATASET, OUTPUT/f'jev_{variant}.json', with_image=WITH_MAP_IMAGE, max_requests=JEV_MAX_REQUESTS, cache_dir=EXPERIMENT/'results/jev_cache'))
else:
    print('Jev не викликався.')

## Після пілота
Збережіть весь каталог OUTPUT. Перед основною кампанією погоджуються кількості знімків, незалежні території, розподіл, набори можливостей корекції та бюджет. Scene_05/07/08/11/21 не є незалежним остаточним тестом.
Навчання, калібрування, фіксація моделей і остаточне оцінювання доступні через CLI, описаний у README.md. run_training відхиляє пілотні дані. Пілотні результати не підтверджують придатність моделі до планування.